# Differentiable Quantum Architecture Search

## Overview

This tutorial demonstrates how to utilize the advanced computational features provided by TensorCircuit such as ``jit`` and ``vmap`` to super efficiently simulate the differentiable quantum architecture search (DQAS) algorithm, where an ensemble of quantum circuits with different structures can be compiled to simulate at the same time.

[WIP note]

## Setup

In [1]:
import numpy as np
import tensorcircuit as tc
import tensorflow as tf

In [2]:
K = tc.set_backend("tensorflow")
ctype, rtype = tc.set_dtype("complex128")

np.random.seed(42)
K.set_random_state(42);

## Problem Description

The task is to find the state preparation circuit for GHZ state $\vert \text{GHZ}_N\rangle = \frac{1}{\sqrt{2}}\left(\vert 0^N\rangle +\vert 1^N\rangle \right)$. We prepare a gate pool with rx0, rx1, ry0, ry1, rz0, rz1, cnot01, cnot10 for the $N=2$ demo. Amongst the eight gates, six are parameterized.

In [3]:
def rx0(theta):
    return K.kron(
        K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._x_matrix, K.eye(2)
    )


def rx1(theta):
    return K.kron(
        K.eye(2), K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._x_matrix
    )


def ry0(theta):
    return K.kron(
        K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._y_matrix, K.eye(2)
    )


def ry1(theta):
    return K.kron(
        K.eye(2), K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._y_matrix
    )


def rz0(theta):
    return K.kron(
        K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._z_matrix, K.eye(2)
    )


def rz1(theta):
    return K.kron(
        K.eye(2), K.cos(theta) * K.eye(2) + 1.0j * K.sin(theta) * tc.gates._z_matrix
    )


def cnot01():
    return K.cast(K.convert_to_tensor(tc.gates._cnot_matrix), ctype)


def cnot10():
    return K.cast(
        K.convert_to_tensor(
            np.array([[1, 0, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0]])
        ),
        ctype,
    )


ops_repr = ["rx0", "rx1", "ry0", "ry1", "rz0", "rz1", "cnot01", "cnot10"]

In [4]:
n, p, ch = 2, 3, 8
# number of qubits, number of layers, size of operation pool

target = tc.array_to_tensor(np.array([1, 0, 0, 1.0]) / np.sqrt(2.0))
# target wavefunction, we here use GHZ2 state as the objective target


def ansatz(params, structures):
    c = tc.Circuit(n)
    params = K.cast(params, ctype)
    structures = K.cast(structures, ctype)
    for i in range(p):
        c.any(
            0,
            1,
            unitary=structures[i, 0] * rx0(params[i, 0])
            + structures[i, 1] * rx1(params[i, 1])
            + structures[i, 2] * ry0(params[i, 2])
            + structures[i, 3] * ry1(params[i, 3])
            + structures[i, 4] * rz0(params[i, 4])
            + structures[i, 5] * rz1(params[i, 5])
            + structures[i, 6] * cnot01()
            + structures[i, 7] * cnot10(),
        )
    s = c.state()
    loss = K.sum(K.abs(target - s))
    return loss


vag1 = K.jit(K.vvag(ansatz, argnums=0, vectorized_argnums=1))

## Probability Ensemble Approach

This approach is more practical and experimental relevant and is the same algorithm described in Ref.1, though we here use advanced vmap to accelerate the simulation of circuits with different structures.

In [5]:
def sampling_from_structure(structures, batch=1):
    prob = K.softmax(K.real(structures), axis=-1)
    return np.array([np.random.choice(ch, p=K.numpy(prob[i])) for i in range(p)])


@K.jit
def best_from_structure(structures):
    return K.argmax(structures, axis=-1)


@K.jit
def nmf_gradient(structures, oh):
    """
    compute the Monte Carlo gradient with respect of naive mean-field probabilistic model
    """
    choice = K.argmax(oh, axis=-1)
    prob = K.softmax(K.real(structures), axis=-1)
    indices = K.transpose(K.stack([K.cast(tf.range(p), "int64"), choice]))
    prob = tf.gather_nd(prob, indices)
    prob = K.reshape(prob, [-1, 1])
    prob = K.tile(prob, [1, ch])

    return tf.tensor_scatter_nd_add(
        tf.cast(-prob, dtype=ctype),
        indices,
        tf.ones([p], dtype=ctype),
    )


nmf_gradient_vmap = K.vmap(nmf_gradient, vectorized_argnums=1)

In [6]:
verbose = False
epochs = 400
batch = 256
lr = tf.keras.optimizers.schedules.ExponentialDecay(0.06, 100, 0.5)
structure_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(0.12))
network_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(lr))
nnp = K.implicit_randn(stddev=0.02, shape=[p, 6], dtype=rtype)
stp = K.implicit_randn(stddev=0.02, shape=[p, 8], dtype=rtype)
avcost1 = 0
for epoch in range(epochs):  # iteration to update strcuture param
    avcost2 = avcost1
    costl = []
    batched_stuctures = K.onehot(
        np.stack([sampling_from_structure(stp) for _ in range(batch)]), num=8
    )
    infd, gnnp = vag1(nnp, batched_stuctures)
    gs = nmf_gradient_vmap(stp, batched_stuctures)  # \nabla lnp
    gstp = [K.cast((infd[i] - avcost2), ctype) * gs[i] for i in range(infd.shape[0])]
    gstp = K.real(K.sum(gstp, axis=0) / infd.shape[0])
    avcost1 = K.sum(infd) / infd.shape[0]
    nnp = network_opt.update(gnnp, nnp)
    stp = structure_opt.update(gstp, stp)

    if epoch % 40 == 0 or epoch == epochs - 1:
        print("----------epoch %s-----------" % epoch)
        print(
            "batched average loss: ",
            np.mean(avcost1),
        )

        if verbose:
            print(
                "strcuture parameter: \n",
                stp.numpy(),
                "\n network parameter: \n",
                nnp.numpy(),
            )

        cand_preset = best_from_structure(stp)
        print("best candidates so far:", [ops_repr[i] for i in cand_preset])
        print(
            "corresponding weights for each gate:",
            [K.numpy(nnp[j, i]) if i < 6 else 0.0 for j, i in enumerate(cand_preset)],
        )

search_structure = K.onehot(best_from_structure(stp), num=ch)
search_weights = K.copy(nnp)

----------epoch 0-----------
batched average loss:  1.0144561278026294
best candidates so far: ['ry1', 'ry0', 'rz1']
corresponding weights for each gate: [np.float64(-0.036889643959831464), np.float64(-0.045930423575561076), np.float64(-0.05586136516270796)]


----------epoch 40-----------
batched average loss:  0.9945892243353422
best candidates so far: ['cnot01', 'cnot01', 'cnot01']
corresponding weights for each gate: [0.0, 0.0, 0.0]


----------epoch 80-----------
batched average loss:  0.009021526329786431
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7765469406363297), 0.0]


----------epoch 120-----------
batched average loss:  0.00033624089511053476
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.789264568040135), 0.0]


----------epoch 160-----------
batched average loss:  0.0012565211088472195
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7872552203567166), 0.0]


...

----------epoch 240-----------
batched average loss:  0.0025613938450768188
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7851708054600723), 0.0]


----------epoch 280-----------
batched average loss:  0.00046227132733540416
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7861842104061324), 0.0]


----------epoch 320-----------
batched average loss:  0.0043531943570100365
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7850287573577476), 0.0]


----------epoch 360-----------
batched average loss:  0.001728741700721904
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7848799846588557), 0.0]


----------epoch 399-----------
batched average loss:  0.0002611930957390207
best candidates so far: ['cnot01', 'ry0', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.7852636308031593), 0.0]


## Directly Optimize the Structure Parameters

Since we are using numerical simulation anyway, we can directly optimize the structure parameter and omit whether the super circuit is unitary or not, this approach can be faster and more reliable for some scenarios.

In [7]:
def ansatz2(params, structures):
    c = tc.Circuit(n)
    params = K.cast(params, ctype)
    structures = K.softmax(structures, axis=-1)
    structures = K.cast(structures, ctype)
    for i in range(p):
        c.any(
            0,
            1,
            unitary=structures[i, 0] * rx0(params[i, 0])
            + structures[i, 1] * rx1(params[i, 1])
            + structures[i, 2] * ry0(params[i, 2])
            + structures[i, 3] * ry1(params[i, 3])
            + structures[i, 4] * rz0(params[i, 4])
            + structures[i, 5] * rz1(params[i, 5])
            + structures[i, 6] * cnot01()
            + structures[i, 7] * cnot10(),
        )
    s = c.state()
    s /= K.norm(s)
    loss = K.sum(K.abs(target - s))
    return loss


vag2 = K.jit(K.value_and_grad(ansatz2, argnums=(0, 1)))

In [8]:
verbose = True
epochs = 700
lr = tf.keras.optimizers.schedules.ExponentialDecay(0.05, 200, 0.5)
structure_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(0.04))
network_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(lr))
nnp = K.implicit_randn(stddev=0.02, shape=[p, 6], dtype=rtype)
stp = K.implicit_randn(stddev=0.02, shape=[p, 8], dtype=rtype)
for epoch in range(epochs):
    infd, (gnnp, gstp) = vag2(nnp, stp)

    nnp = network_opt.update(gnnp, nnp)
    stp = structure_opt.update(gstp, stp)
    if epoch % 70 == 0 or epoch == epochs - 1:
        print("----------epoch %s-----------" % epoch)
        print(
            "batched average loss: ",
            np.mean(infd),
        )
        if verbose:
            print(
                "strcuture parameter: \n",
                stp.numpy(),
                "\n network parameter: \n",
                nnp.numpy(),
            )

        cand_preset = best_from_structure(stp)
        print("best candidates so far:", [ops_repr[i] for i in cand_preset])
        print(
            "corresponding weights for each gate:",
            [K.numpy(nnp[j, i]) if i < 6 else 0.0 for j, i in enumerate(cand_preset)],
        )

----------epoch 0-----------
batched average loss:  1.0050879079544945
strcuture parameter: 
 [[ 0.02886197 -0.01904097 -0.00411362 -0.03629159  0.03929735  0.02925447
   0.07446017  0.05744058]
 [-0.04284884  0.04277968  0.02141616 -0.03786799 -0.01367674 -0.02267738
   0.04154643  0.03172704]
 [ 0.02771259 -0.04723203 -0.03036468  0.02019817  0.02450589  0.06302501
   0.06604171  0.05911277]] 
 network parameter: 
 [[ 0.05078596 -0.03544051  0.03647056  0.03445371  0.0222037   0.03984547]
 [ 0.01666987 -0.06708192  0.05869214  0.03954113  0.05871057  0.04976646]
 [ 0.07201433 -0.03661483  0.02565953  0.0559807   0.06216705  0.04422813]]
best candidates so far: ['cnot01', 'rx1', 'cnot01']
corresponding weights for each gate: [0.0, np.float64(-0.06708191621806722), 0.0]


----------epoch 70-----------
batched average loss:  0.8449109155605046
strcuture parameter: 
 [[-1.12511469 -1.41432752 -0.38104297  0.70794278 -0.40211103 -0.40081366
  -0.27817755 -0.29520255]
 [-0.91571424 -1.08854896 -1.12946239 -0.11235851 -0.97282521 -1.00669768
   1.83539776  0.89557447]
 [-1.44451465 -1.91955314 -0.29387855  0.62966508 -1.09895983 -1.08875152
  -0.39070218  1.48673957]] 
 network parameter: 
 [[ 0.00113795  0.00144792 -0.38505408 -0.93980478 -0.01732954  0.010612  ]
 [-0.02075408 -0.02163527 -0.00630941 -0.39080497  0.01029853  0.00832793]
 [ 0.02649214  0.0032035   0.52153258  0.89184104  0.03084647 -0.05753474]]
best candidates so far: ['ry1', 'cnot01', 'cnot10']
corresponding weights for each gate: [np.float64(-0.9398047799444436), 0.0, 0.0]


----------epoch 140-----------
batched average loss:  0.02960760107426457
strcuture parameter: 
 [[-1.6003747  -1.87910076 -1.28009108  1.40217047 -0.87793092 -0.8773492
  -0.74395648 -0.76098123]
 [-0.61908013 -0.62834538 -1.56723609  0.59796949 -0.5849632  -0.62002455
   3.11235867 -0.40765383]
 [-2.54795737 -3.02508503 -1.16861611  0.85500911 -2.21946662 -2.21258636
  -1.57677407  2.69133259]] 
 network parameter: 
 [[-0.01394824 -0.0043002  -0.22241476 -1.3225966   0.01119868  0.01128775]
 [-0.01393795 -0.02495426  0.08625377 -0.67920697  0.01485004  0.00513701]
 [ 0.04364746 -0.00434622  0.56162766  1.42583448 -0.0140567  -0.06240763]]
best candidates so far: ['ry1', 'cnot01', 'cnot10']
corresponding weights for each gate: [np.float64(-1.3225965999857874), 0.0, 0.0]


----------epoch 210-----------
batched average loss:  0.008087908393836087
strcuture parameter: 
 [[-1.62687517 -1.89565534 -1.29706542  1.42296275 -0.90470594 -0.90412425
  -0.75971491 -0.77673966]
 [-0.74293691 -0.61564524 -1.58500785  0.55720772 -0.65043061 -0.62609165
   3.17600471 -0.42084226]
 [-2.55294199 -3.02920462 -1.19859275  0.79874334 -2.2272991  -2.22149618
  -1.58044693  2.74801826]] 
 network parameter: 
 [[-0.00402064 -0.00274513 -0.24395873 -1.31312418  0.0031124   0.00314326]
 [-0.00488667 -0.02573948  0.08305403 -0.66717573  0.00962805 -0.00391304]
 [ 0.04428737 -0.00436013  0.54782207  1.41615721 -0.00654547 -0.07093285]]
best candidates so far: ['ry1', 'cnot01', 'cnot10']
corresponding weights for each gate: [np.float64(-1.3131241841893735), 0.0, 0.0]


----------epoch 280-----------
batched average loss:  0.00888599648896293
strcuture parameter: 
 [[-1.64443145 -1.91237404 -1.31156412  1.44006817 -0.92274762 -0.92216457
  -0.77643766 -0.79346241]
 [-0.74237796 -0.59460879 -1.59491074  0.53285556 -0.63874892 -0.60861604
   3.20761816 -0.43017774]
 [-2.5583888  -3.03461665 -1.2336967   0.76488765 -2.23294047 -2.22793155
  -1.58493042  2.78590212]] 
 network parameter: 
 [[-0.00471435 -0.00350249 -0.25820009 -1.30910389  0.00382346  0.0038541 ]
 [-0.00357298 -0.02570615  0.08002032 -0.66035349  0.00876335 -0.00208551]
 [ 0.04470103 -0.00449523  0.53571313  1.41641663 -0.01002049 -0.06870404]]
best candidates so far: ['ry1', 'cnot01', 'cnot10']
corresponding weights for each gate: [np.float64(-1.3091038948168738), 0.0, 0.0]


...

----------epoch 420-----------
batched average loss:  0.0025616598339223873
strcuture parameter: 
 [[-1.64085404 -1.90832579 -1.30927711  1.43624759 -0.91878887 -0.91820516
  -0.7723898  -0.78941454]
 [-0.698132   -0.53168286 -1.59242995  0.46785987 -0.57840841 -0.54585531
   3.28355605 -0.45527739]
 [-2.57501244 -3.05105722 -1.27944727  0.7430347  -2.24941797 -2.24441735
  -1.5988181   2.81466025]] 
 network parameter: 
 [[-0.00530495 -0.00234685 -0.26270087 -1.29105497  0.003937    0.00396568]
 [-0.0049589  -0.02518357  0.08240803 -0.64045787  0.01043838 -0.00379299]
 [ 0.04446406 -0.00445327  0.52744974  1.42158236 -0.00900756 -0.07018707]]
best candidates so far: ['ry1', 'cnot01', 'cnot10']
corresponding weights for each gate: [np.float64(-1.2910549742145263), 0.0, 0.0]


----------epoch 490-----------
batched average loss:  0.006584681373880063
strcuture parameter: 
 [[-1.65513865 -1.92267821 -1.3221179   1.45046879 -0.93306382 -0.93247959
  -0.78675258 -0.80377733]
 [-0.68549597 -0.51810676 -1.59763776  0.45320165 -0.56496875 -0.53226073
   3.30197551 -0.46604632]
 [-2.58010819 -3.05611185 -1.30522652  0.71961192 -2.25445281 -2.2494224
  -1.60336785  2.84044522]] 
 network parameter: 
 [[-0.00547436 -0.00237824 -0.2677029  -1.28958778  0.00407717  0.00410582]
 [-0.00526056 -0.02514834  0.08210238 -0.6375599   0.01052311 -0.00367911]
 [ 0.04439003 -0.00444219  0.52311052  1.42120258 -0.00874708 -0.07044293]]
best candidates so far: ['ry1', 'cnot01', 'cnot10']
corresponding weights for each gate: [np.float64(-1.2895877843005277), 0.0, 0.0]


----------epoch 560-----------
batched average loss:  0.0029658286080343184
strcuture parameter: 
 [[-1.65117142 -1.91866916 -1.31455459  1.44613072 -0.92906369 -0.92847935
  -0.78274402 -0.79976877]
 [-0.66718892 -0.49689657 -1.59925564  0.42648657 -0.54401904 -0.51093417
   3.32962685 -0.46769782]
 [-2.57842476 -3.0546351  -1.31479694  0.71126356 -2.253016   -2.24825738
  -1.60108595  2.84914279]] 
 network parameter: 
 [[-0.00626514 -0.00218821 -0.26960519 -1.28482111  0.00452372  0.00455204]
 [-0.00518411 -0.02495143  0.08224593 -0.63248722  0.01104355 -0.00355892]
 [ 0.04453783 -0.00450349  0.52121627  1.41889855 -0.01017279 -0.06910798]]
best candidates so far: ['ry1', 'cnot01', 'cnot10']
corresponding weights for each gate: [np.float64(-1.284821112753881), 0.0, 0.0]


----------epoch 630-----------
batched average loss:  0.012021653401644081
strcuture parameter: 
 [[-1.65572826 -1.92207619 -1.31601263  1.4499811  -0.93441116 -0.93382725
  -0.78573843 -0.80276317]
 [-0.68662427 -0.49261152 -1.60465468  0.40906549 -0.55534275 -0.52523215
   3.3533     -0.47840487]
 [-2.58581243 -3.06263142 -1.3303439   0.69948087 -2.26201023 -2.25826267
  -1.60653589  2.86291997]] 
 network parameter: 
 [[-3.49533429e-03 -1.01346302e-03 -2.71409101e-01 -1.28255106e+00
   4.80599089e-03  4.83414727e-03]
 [-2.91178938e-03 -2.41034147e-02  8.16548327e-02 -6.29905799e-01
   1.20670812e-02 -4.56461229e-03]
 [ 4.51266978e-02 -4.31084427e-03  5.19472234e-01  1.41729477e+00
  -7.44032298e-03 -7.03506486e-02]]
best candidates so far: ['ry1', 'cnot01', 'cnot10']
corresponding weights for each gate: [np.float64(-1.2825510647397063), 0.0, 0.0]


----------epoch 699-----------
batched average loss:  0.001863976915019795
strcuture parameter: 
 [[-1.65404257 -1.92017242 -1.31075301  1.44781477 -0.9326458  -0.93206179
  -0.78379779 -0.80082253]
 [-0.66784443 -0.46012677 -1.60444435  0.37693847 -0.52652551 -0.49491399
   3.39099348 -0.49456071]
 [-2.60241304 -3.07769153 -1.35761351  0.69054528 -2.27716718 -2.27315748
  -1.61773161  2.87586084]] 
 network parameter: 
 [[-0.00594427 -0.00199582 -0.27275929 -1.27877818  0.00424919  0.00427704]
 [-0.00623341 -0.02455677  0.08158688 -0.6259116   0.01081562 -0.00401278]
 [ 0.04417245 -0.004387    0.51745361  1.41683219 -0.0078493  -0.07100341]]
best candidates so far: ['ry1', 'cnot01', 'cnot10']
corresponding weights for each gate: [np.float64(-1.2787781793032564), 0.0, 0.0]


## Final Fine-tune

Continue from the discrete structure and trained weights obtained by the probability ensemble search. Both were saved before the direct structure optimization, so that search does not overwrite the fine-tuning starting point.

In [9]:
chosen_structure = K.reshape(search_structure, [1, p, ch])
nnp = K.copy(search_weights)
initial_loss = ansatz(nnp, search_structure)
print(
    "Selected circuit:",
    [ops_repr[i] for i in K.numpy(K.argmax(search_structure, axis=-1))],
)
print("Loss before fine-tuning:", K.numpy(initial_loss))
chosen_structure

Selected circuit: ['cnot01', 'ry0', 'cnot01']
Loss before fine-tuning: 0.00019025781885084747


<tf.Tensor: shape=(1, 3, 8), dtype=float32, numpy=
array([[[0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0.]]], dtype=float32)>

In [10]:
lr = tf.keras.optimizers.schedules.ExponentialDecay(1e-3, 100, 0.5)
network_opt = tc.backend.optimizer(tf.keras.optimizers.Adam(lr))
verbose = True
epochs = 600
for epoch in range(epochs):
    infd, gnnp = vag1(nnp, chosen_structure)
    nnp = network_opt.update(gnnp, nnp)
    if epoch % 60 == 0 or epoch == epochs - 1:
        print(epoch, "loss: ", K.numpy(infd[0]))

final_loss = ansatz(nnp, search_structure)
print("Loss after fine-tuning:", K.numpy(final_loss))

0 loss:  0.00019025781885084747


60 loss:  0.0001103772316951579


120 loss:  0.0001233379018483749


180 loss:  0.00010164728190031891


240 loss:  3.1655511881667486e-05


...

360 loss:  4.51618538321874e-06


420 loss:  6.157090482572514e-06


480 loss:  2.877099645193226e-07


540 loss:  1.8334828881805265e-06


599 loss:  1.6841730250405362e-06
Loss after fine-tuning: 3.2822604556503876e-06


## References

1. https://arxiv.org/pdf/2010.08561.pdf